# Priority Classification — 75 GitHub Native Issues
## System A (No RAG) vs System B (RAG)

Both systems use the V2 prompt with domain-calibration rules.

- **System A**: Leave-one-out evaluation using only the 75 GitHub native issues as context.
- **System B**: Same prompt + RAG context retrieved via **BGE-base-en-v1.5 + BM25 + RRF** from the Bugzilla XML corpus (𝓑).

**Embedding model**: `BAAI/bge-base-en-v1.5` (consistent with all other task notebooks).
**Retrieval**: Reciprocal Rank Fusion (RRF, k₀=60) over dense cosine similarity and BM25 lexical scores.
**Input**: A single `full-edkII-dd.csv` containing both Bugzilla transferred issues (2,535) and GitHub native issues (75).
Issues with 'bugzilla bug #' in the title are used to build 𝓑 for System B. The 75 native issues are the evaluation set.


## Cell 1 — Install dependencies

In [ ]:
!pip install openai anthropic chromadb sentence-transformers rank_bm25 pandas scikit-learn tqdm -q

## Cell 2 — Configuration

In [ ]:
import pandas as pd
import json
import time
import os
import re
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, balanced_accuracy_score
)

# ── Paths ─────────────────────────────────────────────────────────────────────
DRIVE_CSV   = '/content/drive/MyDrive/priority_classification/data/full-edkII-dd.csv'
CONTENT_CSV = '/content/full-edkII-dd.csv'
GITHUB_CSV  = DRIVE_CSV if os.path.exists(DRIVE_CSV) else CONTENT_CSV
print(f'Using CSV: {GITHUB_CSV}')

# ── Models ────────────────────────────────────────────────────────────────────
OPENAI_MODELS = [
    "gpt-4o-mini",
    "gpt-4o",
    "gpt-4.1-mini",
    "gpt-4.1",
    "gpt-5.4-mini",
    "gpt-5.4",
    "gpt-5.5",
]

CLAUDE_MODELS = [
    "claude-haiku-4-5",
    "claude-sonnet-4-6",
    "claude-opus-4-7",
]

_RESPONSES_API_MODELS = {
    "gpt-5", "gpt-5-mini", "gpt-5.1", "gpt-5.4", "gpt-5.4-mini",
    "gpt-5.4-nano", "gpt-5.4-pro", "gpt-5.5", "gpt-5.5-pro",
    "o3", "o3-mini", "o4-mini",
}

PRIORITY_LABELS = ["low", "medium", "high"]

# ── RAG / retrieval config (System B) ─────────────────────────────────────────
RAG_TOP_K       = 5           # κ: top-k retrieved Bugzilla bugs
EMBED_MODEL     = "BAAI/bge-base-en-v1.5"   # consistent with all other task notebooks
EMBED_BATCH     = 64
RRF_K           = 60          # k₀ constant in RRF formula
CHROMA_PATH     = '/content/drive/MyDrive/priority_classification/chroma_bge'
COLLECTION_NAME = 'bugzilla_priority_bge'

print(f'Config loaded. OpenAI models: {len(OPENAI_MODELS)}, Claude models: {len(CLAUDE_MODELS)}')
print(f'Embedding model : {EMBED_MODEL}')
print(f'RAG top-k (κ)   : {RAG_TOP_K}')
print(f'RRF k₀          : {RRF_K}')


Using CSV: /content/drive/MyDrive/priority_classification/data/full-edkII-dd.csv
Config loaded. OpenAI models: 7, Claude models: 3
Embedding model : BAAI/bge-base-en-v1.5
RAG top-k (κ)   : 5
RRF k₀          : 60


## Cell 3 — API Keys

In [ ]:
from google.colab import userdata
from openai import OpenAI
try:
    import anthropic
except ImportError:
    !pip install -q anthropic
    import anthropic

# ── OpenAI key ────────────────────────────────────────────────
try:
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
    print('OpenAI API key loaded')
except:
    OPENAI_API_KEY = input('Paste your OpenAI API key: ').strip()
    os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

# ── Anthropic key ─────────────────────────────────────────────
try:
    ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY') or ''
    print('Anthropic API key loaded')
except:
    ANTHROPIC_API_KEY = ''
    print('No Anthropic API key — Claude models will be skipped')

# ── Org + project ID for OpenAI opt-out header ────────────────
try:
    OPENAI_ORG_ID     = userdata.get('OPENAI_ORG_ID')     or ''
    OPENAI_PROJECT_ID = userdata.get('OPENAI_PROJECT_ID') or ''
except:
    OPENAI_ORG_ID     = ''
    OPENAI_PROJECT_ID = ''

# ── Build clients ─────────────────────────────────────────────
_openai_headers = {}
if OPENAI_ORG_ID:
    _openai_headers['OpenAI-Organization'] = OPENAI_ORG_ID
if OPENAI_PROJECT_ID:
    _openai_headers['OpenAI-Project'] = OPENAI_PROJECT_ID

openai_client    = OpenAI(api_key=OPENAI_API_KEY, default_headers=_openai_headers)
anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY) if ANTHROPIC_API_KEY else None

# ── Final model list (exclude Claude if no key) ───────────────
ALL_MODELS = OPENAI_MODELS + (CLAUDE_MODELS if anthropic_client else [])
print(f'\nModels to run ({len(ALL_MODELS)} total):')
for m in ALL_MODELS:
    provider = 'Anthropic' if m.startswith('claude') else 'OpenAI'
    print(f'  [{provider}] {m}')

OpenAI API key loaded
Anthropic API key loaded

Models to run (10 total):
  [OpenAI] gpt-4o-mini
  [OpenAI] gpt-4o
  [OpenAI] gpt-4.1-mini
  [OpenAI] gpt-4.1
  [OpenAI] gpt-5.4-mini
  [OpenAI] gpt-5.4
  [OpenAI] gpt-5.5
  [Anthropic] claude-haiku-4-5
  [Anthropic] claude-sonnet-4-6
  [Anthropic] claude-opus-4-7


## Cell 4a — Mount Google Drive & Set Up Directories

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

DRIVE_DIR   = '/content/drive/MyDrive/priority_classification'
DATA_DIR    = f'{DRIVE_DIR}/data'
RESULTS_DIR = f'{DRIVE_DIR}/results'

os.makedirs(DATA_DIR,    exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f'Drive mounted → {DRIVE_DIR}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted → /content/drive/MyDrive/priority_classification


## Cell 4b — Upload Data File

Only needed once — file stays in Drive after first upload.

Upload: `full-edkII-dd.csv`

In [ ]:
from google.colab import files
import shutil, os

uploaded = files.upload()  # upload full-edkII-dd.csv

for fname in uploaded:
    content_path = f'/content/{fname}'
    drive_path   = f'{DATA_DIR}/{fname}'

    # Try to copy to Drive if it's mounted
    if os.path.exists(DATA_DIR):
        shutil.copy(content_path, drive_path)
        print(f'  Saved to Drive : {drive_path}')
    else:
        print(f'  Drive not ready — file available at: {content_path}')
        print(f'  This is fine — Cell 2 config will use /content/ path automatically.')

Saving full-edkII-dd.csv to full-edkII-dd (2).csv
  Saved to Drive : /content/drive/MyDrive/priority_classification/data/full-edkII-dd (2).csv


## Cell 4c — Load GitHub Native Issues

In [ ]:
def load_github_data(path):
    """Load and return only the 75 GitHub native issues from the combined CSV."""
    df_all = pd.read_csv(path)

    # Split: Bugzilla transferred issues have 'bugzilla bug #' in the title
    is_bugzilla   = df_all["Title"].str.contains("bugzilla bug", case=False, na=False)
    df_gh = df_all[~is_bugzilla].reset_index(drop=True)

    # Normalise column names to match the rest of the notebook
    df_gh = df_gh.rename(columns={
        "Issue Number":  "issue_number",
        "Title":         "title",
        "First Assignee":"assignees",
    })

    # No separate body column — use title as fallback
    df_gh["body"] = df_gh["title"]
    df_gh["priority"] = df_gh["priority"].str.strip().str.lower()

    print(f"Total rows in CSV   : {len(df_all)}")
    print(f"Bugzilla transferred: {is_bugzilla.sum()}")
    print(f"GitHub native       : {len(df_gh)}")
    print(f"\nPriority distribution:\n{df_gh['priority'].value_counts().to_string()}\n")
    return df_gh

df = load_github_data(GITHUB_CSV)
df.head(3)

Total rows in CSV   : 2610
Bugzilla transferred: 2535
GitHub native       : 75

Priority distribution:
priority
medium    32
low       28
high      15



,issue_number,title,State,assignees,First Assigned At,Triage Hours,Created At,Closed At,URL,Milestone,Comments,package,priority,type,state,body
0,12141,[Bug]: KeyBoard Backspace is not work in UEFI ...,closed,Damien-Chen,2026-02-26T01:56:19+00:00,454.22,2026-02-07T03:43:04Z,2026-02-28T04:43:51Z,https://github.com/tianocore/edk2/issues/12141,NaN,0,ovmfpkg,low,bug,needs-maintainer-feedback,[Bug]: KeyBoard Backspace is not work in UEFI ...
1,12077,[Bug]: Uncrustify configuration formats DEBUG ...,closed,NaN,NaN,NaN,2026-01-28T19:33:07Z,2026-02-02T18:09:17Z,https://github.com/tianocore/edk2/issues/12077,NaN,2,other,low,bug,needs-maintainer-feedback|needs-owner|needs-tr...,[Bug]: Uncrustify configuration formats DEBUG ...
2,12073,[Bug]: UsbMassStorageDxe: UsbMassReadBlocks la...,closed,NaN,NaN,NaN,2026-01-28T17:19:17Z,2026-01-30T07:02:53Z,https://github.com/tianocore/edk2/issues/12073,NaN,0,mdemodulepkg,medium,bug,needs-maintainer-feedback|needs-triage,[Bug]: UsbMassStorageDxe: UsbMassReadBlocks la...


## Cell 5 — Build RAG Retriever from Bugzilla Corpus (System B only)

Indexes the Bugzilla transferred issues into **ChromaDB** (BGE-base-en-v1.5 dense) and **BM25**.
Retrieval uses **Reciprocal Rank Fusion** (RRF, k₀=60) over both ranked lists, returning the top-κ=5 bugs.

Skip this cell if you only want to run System A.


In [ ]:
import torch
import chromadb
from chromadb import EmbeddingFunction, Documents, Embeddings
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from tqdm.notebook import tqdm


# ── Helpers ───────────────────────────────────────────────────────────────────
def get_device():
    if torch.cuda.is_available():   return 'cuda'
    if hasattr(torch.backends,'mps') and torch.backends.mps.is_available(): return 'mps'
    return 'cpu'

def tokenise(text):
    text = re.sub(r'[^a-zA-Z0-9_]', ' ', (text or '').lower())
    return [t for t in text.split() if len(t) > 1]

def rrf(ranked_a, ranked_b, k=RRF_K):
    """Reciprocal Rank Fusion — combines two ranked lists."""
    scores = {}
    for rank, doc_id in enumerate(ranked_a, 1):
        scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank)
    for rank, doc_id in enumerate(ranked_b, 1):
        scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank)
    return sorted(scores, key=scores.__getitem__, reverse=True)


class BGEEmbeddingFunction(EmbeddingFunction):
    def __init__(self):
        device = get_device()
        print(f'[retriever] Loading {EMBED_MODEL} on {device}...')
        self._model = SentenceTransformer(EMBED_MODEL, device=device)
        print('[retriever] Model loaded.')

    def __call__(self, input: Documents) -> Embeddings:
        return self._model.encode(
            list(input), batch_size=EMBED_BATCH,
            normalize_embeddings=True, show_progress_bar=False
        ).tolist()


# ── Parse & split CSV ─────────────────────────────────────────────────────────
def _clean(text):
    text = re.sub(r'https?://\S+', '<URL>', text or '')
    return re.sub(r'\s+', ' ', text).strip()

df_all = pd.read_csv(GITHUB_CSV)
is_bugzilla = df_all['Title'].str.contains('bugzilla bug', case=False, na=False)
df_bz = df_all[is_bugzilla].reset_index(drop=True)
df_gh = df_all[~is_bugzilla].reset_index(drop=True)
print(f'Bugzilla transferred : {len(df_bz)} issues')
print(f'GitHub native        : {len(df_gh)} issues')

# Build list of dicts for BM25 + ChromaDB
bz_docs = []
for _, row in df_bz.iterrows():
    title    = str(row.get('Title', '') or '')
    pkg      = str(row.get('package', '') or '')
    priority = str(row.get('priority', '') or '').strip().lower()
    iid      = str(int(row['Issue Number']))
    text     = _clean(f'Title: {title} Package: {pkg}')
    bz_docs.append({'id': iid, 'title': title, 'package': pkg,
                    'priority': priority, 'text': text})

# Filter to bugs with known priority labels
bz_pri = [b for b in bz_docs if b['priority'] in ('low', 'medium', 'high')]
print(f'Bugzilla bugs with known priority (𝓑^𝒫): {len(bz_pri)}')

# ── Build ChromaDB (BGE dense) + BM25 indexes ─────────────────────────────────
os.makedirs(CHROMA_PATH, exist_ok=True)
_embed_fn   = BGEEmbeddingFunction()
_chroma     = chromadb.PersistentClient(path=CHROMA_PATH)

# Delete and recreate collection to avoid corrupted index
try:
    _chroma.delete_collection(COLLECTION_NAME)
    print('[retriever] Deleted existing collection.')
except:
    pass

_collection = _chroma.get_or_create_collection(
    COLLECTION_NAME, embedding_function=_embed_fn,
    metadata={'hnsw:space': 'cosine'}
)

print(f'[retriever] Indexing {len(bz_pri)} Bugzilla bugs...')
for i in tqdm(range(0, len(bz_pri), EMBED_BATCH), desc='BGE index'):
    batch = bz_pri[i:i + EMBED_BATCH]
    _collection.add(
        ids       = [b['id']   for b in batch],
        documents = [b['text'] for b in batch],
        metadatas = [{'title': b['title'], 'package': b['package'],
                      'priority': b['priority']} for b in batch],
    )
print(f'[retriever] Indexed {_collection.count()} docs.')

_bz_bm25_ids = [b['id']   for b in bz_pri]
_bz_bm25     = BM25Okapi([tokenise(b['text']) for b in bz_pri])
_bz_by_id    = {b['id']: b for b in bz_pri}
print('[retriever] BM25 index ready.')


# ── Retrieval function: RRF over BGE + BM25 ───────────────────────────────────
def retrieve_rag_context(test_row, top_k=RAG_TOP_K):
    """Return top-k Bugzilla bugs via RRF(Dense, BM25) for priority context.

    Implements: 𝓡_k = RRF(Dense(b_k, 𝓑^𝒫), BM25(b_k, 𝓑^𝒫))_{1:κ}
    """
    title = str(test_row.get('title', '') or '')
    pkg   = str(test_row.get('package', '') or '')
    query = _clean(f'Title: {title} Package: {pkg}')

    # Dense retrieval (BGE cosine)
    fetch      = min(top_k * 4, _collection.count())
    dense_res  = _collection.query(query_texts=[query], n_results=fetch,
                                   include=['distances'])
    dense_ids  = dense_res['ids'][0]

    # Sparse retrieval (BM25)
    bm25_scores = _bz_bm25.get_scores(tokenise(query))
    sparse_ids  = [_bz_bm25_ids[i]
                   for i in sorted(range(len(bm25_scores)),
                                   key=lambda x: bm25_scores[x], reverse=True)]

    # Fuse
    fused = rrf(dense_ids, sparse_ids)[:top_k]
    return [_bz_by_id[i] for i in fused if i in _bz_by_id]


print(f'RAG retriever ready (BGE-base-en-v1.5 + BM25 + RRF, κ={RAG_TOP_K}).')

Bugzilla transferred : 2535 issues
GitHub native        : 75 issues
Bugzilla bugs with known priority (𝓑^𝒫): 2534
[retriever] Loading BAAI/bge-base-en-v1.5 on cpu...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[retriever] Model loaded.
[retriever] Deleted existing collection.
[retriever] Indexing 2534 Bugzilla bugs...


BGE index:   0%|          | 0/40 [00:00<?, ?it/s]

[retriever] Indexed 2534 docs.
[retriever] BM25 index ready.
RAG retriever ready (BGE-base-en-v1.5 + BM25 + RRF, κ=5).


## Cell 6 — Issue Formatting & Prompt Builders

In [ ]:
def format_issue(row, include_priority=True):
    """Format a single issue row as a readable string."""
    body_preview = str(row.get("body", ""))[:300].replace("\n", " ").strip()
    pkg = str(row.get("package", "")).strip()
    pkg = pkg if pkg and pkg != "nan" else "unknown"
    text = (
        f"Title   : {row['title']}\n"
        f"Package : {pkg}\n"
        f"Body    : {body_preview}"
    )
    if include_priority:
        text += f"\nPriority: {row['priority']}"
    return text


# ── Shared domain-calibration system prompt (V2) ──────────────────────────────
DOMAIN_RULES = """\
You are a bug priority classifier for the tianocore/edk2 UEFI firmware repository.

PRIORITY DEFINITIONS:
- high   : Crashes, data corruption, security vulnerabilities that directly
           apply to the UEFI threat model, or build failures that block ALL
           configurations for ALL users.
- medium : Functional bugs with workarounds, build failures limited to a
           specific compiler or toolchain, missing features required for spec
           compliance, race conditions that may cause instability.
- low    : Cosmetic issues, code quality improvements, warnings that do not
           block builds, missing optional features, documentation changes,
           bugs in non-default configurations.

DOMAIN KNOWLEDGE RULES for edk2 priority triage:

1. CVE / security updates are NOT automatically high. UEFI firmware has a
   different threat model than an OS. If the vulnerable code path in a library
   (e.g. OpenSSL) is not reachable during UEFI execution, the patch priority
   reflects maintenance urgency, not security severity.

2. Build failures severity depends on scope. A failure that blocks ALL users
   regardless of compiler or configuration is high. A failure limited to one
   specific compiler (e.g. VS2022 only, CLANG only) or one toolchain variant
   (e.g. CLANGPDB, CLANGDWARF) is medium. A warning that does not block the
   build at all is low.

3. UI and display bugs are low unless they completely prevent the user from
   interacting with the firmware (e.g. no input accepted at all).

4. Missing feature additions or spec compliance gaps are medium if the spec
   is published and widely adopted, low if the feature is optional or cosmetic.

5. ALWAYS use the training examples as your primary signal. If a similar bug
   in the examples was labeled low, do not escalate without a concrete reason
   specific to this bug that is not present in the example.
"""

JSON_INSTRUCTION = """\
Before predicting, scan the examples above for the most similar issue — same
package, same bug type. Use that example's label as your anchor. Only deviate
if there is a concrete, specific reason this issue is clearly more or less
severe than the similar example.

Only choose from: low, medium, high

Respond ONLY with a JSON object, nothing else — no prose before or after:
{"predicted_priority": "<low|medium|high>", "confidence": "<high|medium|low>", "reason": "<one sentence>"}"""


# ── System A prompt: leave-one-out from GitHub 75 ────────────────────────────
def build_prompt_system_a(train_df, test_row):
    """System A — no RAG. Context = other 74 GitHub native issues."""
    examples   = "\n\n".join(format_issue(r) for _, r in train_df.iterrows())
    test_issue = format_issue(test_row, include_priority=False)
    return f"""{DOMAIN_RULES}
---
Past GitHub issues and their priority labels:

{examples}

---
New issue to classify:

{test_issue}

{JSON_INSTRUCTION}"""


# ── Helper: format retrieved Bugzilla bugs for the prompt ────────────────────
def _format_rag_context(bugs):
    """Format retrieved Bugzilla bugs as a prompt string."""
    return "\n\n".join(
        f"Title   : {b['title']}\nPackage : {b['package']}\nPriority: {b['priority']}"
        for b in bugs
    )


# ── System B prompt: leave-one-out GitHub 75 + RAG from Bugzilla ─────────────
def build_prompt_system_b(train_df, test_row, collection=None):
    """System B — RAG enabled.
    R_k = RRF(Dense(b_k, B^P), BM25(b_k, B^P))_{1:k}  (k=5, k0=60)
    Context = top-k Bugzilla bugs (priority-labelled) + GitHub 74 LOO examples.
    """
    github_examples = "\n\n".join(format_issue(r) for _, r in train_df.iterrows())
    rag_bugs        = retrieve_rag_context(test_row)   # BGE+BM25+RRF, defined in Cell 5
    rag_context     = _format_rag_context(rag_bugs)
    test_issue      = format_issue(test_row, include_priority=False)

    return f"""{DOMAIN_RULES}
---
SIMILAR HISTORICAL BUGS from Bugzilla (retrieved for context):

{rag_context}

---
Past GitHub issues and their priority labels:

{github_examples}

---
New issue to classify:

{test_issue}

{JSON_INSTRUCTION}"""


print("Prompt builders ready.")

Prompt builders ready.


## Cell 7 — Model Inference & Evaluation Loop

In [ ]:
def call_model(model_id, prompt):
    """Unified model call — routes to OpenAI Chat, OpenAI Responses API, or Anthropic."""
    is_claude        = model_id.startswith('claude')
    use_responses    = any(model_id.startswith(m) for m in _RESPONSES_API_MODELS)

    for attempt in range(3):
        try:
            if is_claude:
                resp = anthropic_client.messages.create(
                    model      = model_id,
                    max_tokens = 200,
                    messages   = [{"role": "user", "content": prompt}],
                )
                raw = resp.content[0].text
            elif use_responses:
                resp = openai_client.responses.create(
                    model         = model_id,
                    input         = prompt,
                    extra_headers = _openai_headers,
                )
                raw = resp.output_text
            else:
                resp = openai_client.chat.completions.create(
                    model         = model_id,
                    messages      = [{"role": "user", "content": prompt}],
                    temperature   = 0.0,
                    max_tokens    = 200,
                    extra_headers = _openai_headers,
                )
                raw = resp.choices[0].message.content

            # Parse JSON
            raw   = raw.replace('```json', '').replace('```', '').strip()
            start = raw.find('{')
            end   = raw.rfind('}') + 1
            if start != -1 and end > start:
                raw = raw[start:end]
            return json.loads(raw)

        except Exception as e:
            print(f'  [attempt {attempt+1}] {type(e).__name__}: {e}')
            if attempt < 2:
                time.sleep(2)
    return None


def evaluate_system(df, model_id, system='A', collection=None):
    results = []
    correct = 0
    total   = len(df)
    tag     = f'{model_id} [System {system}]'

    print(f"\n{'='*80}")
    print(f'  {tag}  |  {total} issues')
    print(f"{'='*80}\n")

    for i, (_, test_row) in enumerate(df.iterrows()):
        t0       = time.time()
        train_df = df.drop(df[df['issue_number'] == test_row['issue_number']].index)
        actual   = str(test_row['priority']).strip().lower()

        if system == 'A':
            prompt = build_prompt_system_a(train_df, test_row)
        else:
            prompt = build_prompt_system_b(train_df, test_row)  # retriever is module-level

        result = call_model(model_id, prompt)
        if result is None:
            print(f'  ✗ Failed on #{test_row["issue_number"]} — skipping')
            continue

        predicted  = result.get('predicted_priority', '').strip().lower()
        confidence = result.get('confidence', '')
        reason     = result.get('reason', '')
        match      = predicted == actual
        if match:
            correct += 1

        elapsed = round(time.time() - t0, 2)
        print(f'Issue #{test_row["issue_number"]} ({i+1}/{total})  [{elapsed}s]')
        print(f'  Title     : {str(test_row["title"])[:70]}')
        print(f'  Actual    : {actual}')
        print(f'  Predicted : {predicted}')
        print(f'  Confidence: {confidence}')
        print(f'  Reason    : {reason}')
        print(f'  Match     : {"✓" if match else "✗"}\n')

        results.append({
            'system':          f'System {system}',
            'model':           model_id,
            'issue_number':    test_row['issue_number'],
            'title':           test_row['title'],
            'actual':          actual,
            'predicted':       predicted,
            'confidence':      confidence,
            'reason':          reason,
            'correct':         match,
            'elapsed_seconds': elapsed,
        })
        time.sleep(0.3)

    evaluated = len(results)
    skipped   = total - evaluated
    print(f"{'='*80}")
    print(f'  [{tag}] Accuracy: {correct}/{evaluated} = {correct/evaluated*100:.1f}%  ({skipped} skipped)')
    print(f"{'='*80}")
    return results


print('Evaluation function ready.')

Evaluation function ready.


## Cell 8 — Metrics & Comparison

In [ ]:
def compute_metrics(results, label):
    """Compute and print full metrics for one system/model combination."""
    clean     = [r for r in results if r["predicted"] not in ("", None)]
    actuals   = [r["actual"]    for r in clean]
    predicted = [r["predicted"] for r in clean]
    n         = len(clean)

    acc      = accuracy_score(actuals, predicted)
    bal_acc  = balanced_accuracy_score(actuals, predicted)
    macro_p  = precision_score(actuals, predicted, average="macro",    zero_division=0, labels=PRIORITY_LABELS)
    macro_r  = recall_score(   actuals, predicted, average="macro",    zero_division=0, labels=PRIORITY_LABELS)
    macro_f1 = f1_score(       actuals, predicted, average="macro",    zero_division=0, labels=PRIORITY_LABELS)
    wgt_p    = precision_score(actuals, predicted, average="weighted", zero_division=0, labels=PRIORITY_LABELS)
    wgt_r    = recall_score(   actuals, predicted, average="weighted", zero_division=0, labels=PRIORITY_LABELS)
    wgt_f1   = f1_score(       actuals, predicted, average="weighted", zero_division=0, labels=PRIORITY_LABELS)

    print(f"\n{'='*70}")
    print(f"  Metrics — {label}  ({n}/{len(results)} evaluated)")
    print(f"{'='*70}")
    print(classification_report(actuals, predicted, labels=PRIORITY_LABELS, zero_division=0))
    print(f"  {'Metric':<20} {'Macro':>8}  {'Weighted':>10}")
    print(f"  {'-'*40}")
    print(f"  {'Precision':<20} {macro_p:>8.3f}  {wgt_p:>10.3f}")
    print(f"  {'Recall':<20} {macro_r:>8.3f}  {wgt_r:>10.3f}")
    print(f"  {'F1 Score':<20} {macro_f1:>8.3f}  {wgt_f1:>10.3f}")
    print(f"  {'Accuracy':<20} {acc:>8.3f}")
    print(f"  {'Balanced Accuracy':<20} {bal_acc:>8.3f}")
    print(f"{'='*70}\n")

    return {
        "label":    label,
        "n":        n,
        "accuracy": round(acc,      3),
        "bal_acc":  round(bal_acc,  3),
        "macro_p":  round(macro_p,  3),
        "macro_r":  round(macro_r,  3),
        "macro_f1": round(macro_f1, 3),
        "wgt_p":    round(wgt_p,    3),
        "wgt_r":    round(wgt_r,    3),
        "wgt_f1":   round(wgt_f1,   3),
    }


def print_comparison(summary_rows):
    print(f"\n{'='*90}")
    print("  FINAL COMPARISON — System A vs System B (all models)")
    print(f"{'='*90}")
    header = (
        f"  {'Config':<30} {'N':>5}  {'Acc':>6}  {'BalAcc':>7}  "
        f"{'Mac-P':>7}  {'Mac-R':>7}  {'Mac-F1':>8}  {'Wgt-F1':>8}"
    )
    print(header)
    print(f"  {'-'*92}")
    for s in summary_rows:
        print(
            f"  {s['label']:<30} {s['n']:>5}  {s['accuracy']:>6.3f}  {s.get('bal_acc',0):>7.3f}  "
            f"{s['macro_p']:>7.3f}  {s['macro_r']:>7.3f}  {s['macro_f1']:>8.3f}  {s['wgt_f1']:>8.3f}"
        )
    print(f"{'='*90}\n")


print("Metrics functions ready.")

Metrics functions ready.


## Cell 9 — Run System A (No RAG)

Runs leave-one-out evaluation for all models using only the 75 GitHub issues as context.

In [ ]:
import time

all_results_a = []
summary_a     = []
start         = time.time()

for model_id in ALL_MODELS:
    results = evaluate_system(df, model_id, system='A', collection=None)
    all_results_a.extend(results)
    row = compute_metrics(results, f'{model_id} — System A')
    summary_a.append(row)

mins, secs = divmod(time.time() - start, 60)
print(f'System A complete — {int(mins)}m {secs:.1f}s')

Streaming output truncated to the last 5000 lines.
  Confidence: high
  Reason    : Build failure limited to a specific compiler (GCC 16) with -Werror flag is similar to other compiler-specific build failures labeled medium.
  Match     : ✗

Issue #11965 (7/75)  [4.57s]
  Title     : [Bug]: qemu-kvm GUEST start failed on RISC-V
  Actual    : medium
  Predicted : medium
  Confidence: high
  Reason    : Similar ovmfpkg RISC-V support issues like 'TCG2/TPM2 support on RISC-V does not work' are labeled medium, and guest start failure on RISC-V is a functional bug likely with workarounds or platform-specific impact.
  Match     : ✓

Issue #11948 (8/75)  [1.34s]
  Title     : [Bug]: Mismatch in the value of LAST_ATTEMPT_STATUS_ERROR_UNSUCCESSFUL
  Actual    : low
  Predicted : low
  Confidence: high
  Reason    : A very similar issue with the same package and topic about LAST_ATTEMPT_STATUS_ERROR_UNSUCCESSFUL_VENDOR_RANGE_MAX was labeled low, indicating this is a minor spec compliance or cod

## Cell 10 — Run System B (RAG)

Same prompt + Bugzilla RAG context retrieved per issue.
**Requires Cell 5 to have been run first.**

In [ ]:
all_results_b = []
summary_b     = []
start         = time.time()

for model_id in ALL_MODELS:
    results = evaluate_system(df, model_id, system='B')  # collection not needed; retriever is module-level
    all_results_b.extend(results)
    row = compute_metrics(results, f'{model_id} — System B')
    summary_b.append(row)

mins, secs = divmod(time.time() - start, 60)
print(f'System B complete — {int(mins)}m {secs:.1f}s')

Streaming output truncated to the last 5000 lines.
  Confidence: high
  Reason    : A build failure blocking the ShellPkg for GCC 16 with -Werror is similar to prior high priority build failures in shellpkg that block builds for all users with a specific compiler.
  Match     : ✗

Issue #11965 (7/75)  [5.54s]
  Title     : [Bug]: qemu-kvm GUEST start failed on RISC-V
  Actual    : medium
  Predicted : low
  Confidence: high
  Reason    : Similar past ovmfpkg guest start or emulation failures on RISC-V or QEMU/KVM have been classified as low priority.
  Match     : ✗

Issue #11948 (8/75)  [4.59s]
  Title     : [Bug]: Mismatch in the value of LAST_ATTEMPT_STATUS_ERROR_UNSUCCESSFUL
  Actual    : low
  Predicted : low
  Confidence: high
  Reason    : A very similar past bug with the same package and issue type was labeled low, indicating this is a minor spec compliance or documentation mismatch.
  Match     : ✓

Issue #11895 (9/75)  [2.81s]
  Title     : [Bug]: /EmulatorPkg/Unix/Host/Host.

## Cell 11 — Compare A vs B & Save Results

In [ ]:
# Print side-by-side comparison
print_comparison(summary_a + summary_b)

# Save detailed results to Google Drive
RESULTS_DIR = '/content/drive/MyDrive/priority_classification/results'
results_path = f'{RESULTS_DIR}/priority_github75_results.csv'
summary_path = f'{RESULTS_DIR}/priority_github75_summary.csv'

all_results = all_results_a + all_results_b
pd.DataFrame(all_results).to_csv(results_path, index=False)
pd.DataFrame(summary_a + summary_b).to_csv(summary_path, index=False)

print(f'✓ Detailed results → {results_path}')
print(f'✓ Summary table    → {summary_path}')


  FINAL COMPARISON — System A vs System B (all models)
  Config                             N     Acc   BalAcc    Mac-P    Mac-R    Mac-F1    Wgt-F1
  --------------------------------------------------------------------------------------------
  gpt-4o-mini — System A            75   0.613    0.571    0.629    0.571     0.579     0.604
  gpt-4o — System A                 75   0.587    0.529    0.561    0.529     0.534     0.578
  gpt-4.1-mini — System A           75   0.600    0.552    0.576    0.552     0.556     0.594
  gpt-4.1 — System A                75   0.640    0.609    0.613    0.609     0.609     0.643
  gpt-5.4-mini — System A           75   0.680    0.622    0.633    0.622     0.624     0.674
  gpt-5.4 — System A                75   0.680    0.623    0.652    0.623     0.627     0.668
  gpt-5.5 — System A                75   0.627    0.564    0.605    0.564     0.568     0.611
  claude-haiku-4-5 — System A       75   0.600    0.584    0.613    0.584     0.585     0.599
  c

## Cell 12 — Error Analysis: Consistent Misclassifications

Shows issues where System A and System B both predicted the wrong label — these are the hardest cases.

In [ ]:
df_a = pd.DataFrame(all_results_a)
df_b = pd.DataFrame(all_results_b)

# Merge on issue_number and model
merged = df_a.merge(
    df_b[["issue_number", "model", "predicted", "correct"]],
    on=["issue_number", "model"],
    suffixes=("_A", "_B")
)

# Both wrong
both_wrong = merged[(merged["correct_A"] == False) & (merged["correct_B"] == False)]
print(f"Issues wrong in BOTH System A and B: {len(both_wrong)}\n")
cols = ["issue_number", "model", "title", "actual", "predicted_A", "predicted_B"]
print(both_wrong[cols].to_string(index=False))

print()

# RAG helped (A wrong, B correct)
rag_helped = merged[(merged["correct_A"] == False) & (merged["correct_B"] == True)]
print(f"Issues where RAG helped (A wrong → B correct): {len(rag_helped)}")
print(rag_helped[cols].to_string(index=False))

print()

# RAG hurt (A correct, B wrong)
rag_hurt = merged[(merged["correct_A"] == True) & (merged["correct_B"] == False)]
print(f"Issues where RAG hurt (A correct → B wrong): {len(rag_hurt)}")
print(rag_hurt[cols].to_string(index=False))

Issues wrong in BOTH System A and B: 214

 issue_number             model                                                                                                                                                                                                            title actual predicted_A predicted_B
        12067       gpt-4o-mini                                                                                                                                             [Bug]: MdeModulePkg/HiiDatabaseDxe: Parser issue in GetNameElement() medium         low         low
        12043       gpt-4o-mini                                                                                                                                                           [Bug]: edk2:Shell won't build with GCC 16 with -Werror    low      medium      medium
        11842       gpt-4o-mini                                                                                                               